# Can a small model pass a test?

This notebook asks a fun question: **can a small browser model answer SAT-style multiple-choice questions?**

The point is not to prove that the model is brilliant. In fact, small browser models will often do poorly. That is part of the lesson. You will see where they make reasonable guesses, where they get confused, and how scoring works.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `re` finds patterns in text (these patterns are called **regular expressions**).
- `pandas` (as `pd`) makes and filters tables.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).
- `Markdown` shows formatted text, like a report card, as cell output.

In [ ]:
import json
import re
import pandas as pd
import pyjs
from IPython.display import display, Markdown

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

- `engine` is the part of WebLLM that actually runs a model. It starts as `None` (Python for "nothing yet") and gets created the first time we need it.
- `loaded_model` remembers the name of the model that is currently loaded, so later steps can check that one is ready.
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model()` gets a model ready to run. If the model is not saved in your browser yet, it downloads it first.

This is the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise

print("Model loader is ready.")

## Step 2: Load a model

A small model makes the experiment more honest. It may score poorly, but that is the point. Later you can paste in another model name from notebook 01 and compare scores.

In [ ]:
MODEL_NAME = "gemma3-1b-it-q4f16_1-MLC"

await start_model(MODEL_NAME)
loaded_model = MODEL_NAME
print(f"{loaded_model} is loaded and ready.")

## Step 3: Build the question prompt

Every question is sent in the same format, so the model's answers are easy to compare and grade.

- `build_prompt()` puts the passage (if there is one), the question and the choices into one message, and asks the model to end with a line like `Final answer: B`.
- The **system message** tells the model how to behave: show its reasoning, then give that final line.

Look at the printed example to see exactly what the model receives.

In [ ]:
SYSTEM = "You are a careful tutor. Show your reasoning, then end with exactly one line in the form Final answer: X."


def build_prompt(question_text, choices_text, passage_text=""):
    if passage_text:
        body = "Passage: " + passage_text + "\n\nQuestion: " + question_text
    else:
        body = "Question: " + question_text
    return (
        "Here is an SAT-style multiple-choice question.\n\n"
        + body
        + "\n\nChoices: " + choices_text + "\n\n"
        + "Think carefully. On the very last line of your response, write exactly: Final answer: X where X is A, B, C, or D."
    )


print(build_prompt("If 3x + 5 = 14, what is the value of x?", "A) 2\nB) 3\nC) 4\nD) 5"))

## Step 4: Ask a question and read the answer letter

`answer_question()` sends one question and returns the model's full reply text plus the letter it chose.

- `temperature` is `0.0` and `seed` is fixed, so the model answers the same way each time. That makes the test fair to repeat.
- A **regular expression** (a text pattern, from `re`) looks for `Final answer:` followed by A, B, C or D. If the model never writes that line, the letter is left blank (`""`).

In [ ]:
async def answer_question(question_text, choices_text, passage_text=""):
    """Return (full reply text, answer letter) for one question."""
    request = {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": build_prompt(question_text, choices_text, passage_text)},
        ],
        "temperature": 0.0,
        "seed": 7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    reply = py_obj(await engine.chat.completions.create(js_obj(request)))
    text = reply["choices"][0]["message"]["content"]
    match = re.search(r"[Ff]inal answer:\s*([A-D])", text)
    letter = match.group(1).upper() if match else ""
    return text, letter


print("answer_question() is ready.")

### 4a: Warm up with one question

A simple algebra question, so you can see the reply and the letter we pulled out of it.

In [ ]:
WARMUP_QUESTION = "If 3x + 5 = 14, what is the value of x?"
WARMUP_CHOICES = "A) 2\nB) 3\nC) 4\nD) 5"

text, letter = await answer_question(WARMUP_QUESTION, WARMUP_CHOICES)
print(text)
print()
print("Answer letter:", letter or "(none found)")

## Step 5: Load the PineSAT question bank

This notebook comes with a **dataset of 100 PineSAT questions** stored in `data/pinesat_100_questions.json`, so it keeps working even if the PineSAT website changes or disappears.

The file is a saved copy of 100 questions from the PineSAT question bank: 50 English and 50 Math. It also keeps the difficulty, **domain** (topic, like Algebra), and answer explanation for each question. The table shows how many questions there are of each kind, then the first few questions.

In [ ]:
DATASET_PATH = "data/pinesat_100_questions.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

questions = pd.DataFrame(dataset["questions"])
print(f"Loaded {len(questions)} PineSAT questions from {DATASET_PATH}")
print("Source:", dataset["source_archive"])
print()
print(questions.groupby(["section", "difficulty"]).size())

questions[["section", "difficulty", "domain", "question", "answer"]].head(12)

## Step 6: Build a practice set

Choose a section, a difficulty and how many questions to try. The model runs on your own computer, so start small: two to five questions is a good first test.

The cell keeps only the questions that match, then picks `NUM_QUESTIONS` of them at random.

In [ ]:
SECTION = "English"        # "English" or "Math"
DIFFICULTY = "Easy"        # "Easy", "Medium" or "Hard"
NUM_QUESTIONS = 2

matching = questions[(questions["section"] == SECTION) & (questions["difficulty"] == DIFFICULTY)]

if len(matching) == 0:
    print("No questions match those settings. Check the spelling of SECTION and DIFFICULTY.")
else:
    practice_set = matching.sample(min(NUM_QUESTIONS, len(matching))).to_dict("records")
    print(f"Built a practice set of {len(practice_set)} questions.")
    for number, item in enumerate(practice_set, start=1):
        print()
        print(f"Q{number}: {item['question']}")
        if item["passage"]:
            print("Passage:", item["passage"])
        print("Choices:", item["choices"])
        print("Correct answer:", item["answer"])

## Step 7: Let the model take the mini test

This asks each question in the practice set, records the model's letter, and checks it against the correct answer. It can take a minute.

In [ ]:
results = []
for item in practice_set:
    text, letter = await answer_question(item["question"], item["choices"], item["passage"])
    results.append({
        "section": item["section"],
        "difficulty": item["difficulty"],
        "domain": item["domain"],
        "question": item["question"],
        "correct_answer": item["answer"],
        "model_guess": letter,
        "correct": letter == item["answer"],
        "raw_reply": text,
    })

results_table = pd.DataFrame(results)
results_table[["section", "difficulty", "domain", "correct_answer", "model_guess", "correct"]]

## Step 8: Read the model's full replies

Sometimes the most interesting part is not whether the answer is right, but how the model explained itself.

In [ ]:
for number, row in enumerate(results, start=1):
    print(f"Question {number}")
    print(row["question"])
    print()
    print(row["raw_reply"])
    print()
    print("Model's answer:", row["model_guess"] or "(none found)")
    print("Correct answer:", row["correct_answer"])
    print("Correct?", row["correct"])
    print("\n" + "=" * 70 + "\n")

## Step 9: Score the model

Small browser models often miss several questions. That is useful evidence, not a failure of the notebook.

### 9a: A scoring table

**Accuracy** is the percent of questions answered correctly. We work it out overall, and then for each section, difficulty and domain, so you can see where the model did better or worse. `groupby` splits the results into groups (for example, one group per domain) so we can count each one.

In [ ]:
def score_row(group_name, label, rows):
    return {
        "group": group_name,
        "label": label,
        "correct_answers": int(rows["correct"].sum()),
        "total_questions": len(rows),
    }


score_rows = [score_row("Overall", "All questions", results_table)]
for column, group_name in [("section", "Section"), ("difficulty", "Difficulty"), ("domain", "Domain")]:
    for label, rows in results_table.groupby(column, sort=True):
        score_rows.append(score_row(group_name, label, rows))

score_table = pd.DataFrame(score_rows)
score_table["accuracy"] = (100 * score_table["correct_answers"] / score_table["total_questions"]).round(1)
score_table

### 9b: A model report card

This turns the numbers into a short plain-language summary: a score, a letter grade, how many answers were blank (the model never wrote `Final answer:`), and the strongest and weakest areas in this run.

In [ ]:
overall = score_table.iloc[0]
accuracy = float(overall["accuracy"])
blank_answers = int((results_table["model_guess"] == "").sum())

if accuracy >= 90:
    grade, verdict = "A", "This model was unusually strong on this mini test."
elif accuracy >= 80:
    grade, verdict = "B", "This model did well, with some noticeable mistakes."
elif accuracy >= 70:
    grade, verdict = "C", "This model was mixed: some solid answers, some clear misses."
elif accuracy >= 60:
    grade, verdict = "D", "This model struggled and would need careful checking by a human."
else:
    grade, verdict = "F", "This model had a hard time with this question set."

details = score_table[score_table["group"] != "Overall"]
strongest = details.sort_values(["accuracy", "total_questions"], ascending=False).iloc[0]
weakest = details.sort_values(["accuracy", "total_questions"], ascending=[True, False]).iloc[0]

report = f"""
## Model report card

- **Model:** {loaded_model}
- **Score:** {int(overall["correct_answers"])} out of {int(overall["total_questions"])}
- **Accuracy:** {accuracy:.1f}%
- **Letter grade:** {grade}
- **Blank answers:** {blank_answers}
- **Strongest area in this run:** {strongest["group"]}: {strongest["label"]} ({strongest["accuracy"]:.1f}%)
- **Weakest area in this run:** {weakest["group"]}: {weakest["label"]} ({weakest["accuracy"]:.1f}%)

{verdict}
"""

display(Markdown(report))

## Step 10: Optional: explore domains and explanations

The question bank includes a **domain** for each question, such as Algebra or Information and Ideas, plus a written explanation of the correct answer. Put a domain name in `DOMAIN` to see just those questions, or leave it as `""` to see the first ten of all of them.

In [ ]:
DOMAIN = ""

shown = questions
if DOMAIN:
    shown = shown[shown["domain"] == DOMAIN]

print("Domains:", ", ".join(sorted(questions["domain"].unique())))
print("Questions shown:", min(len(shown), 10), "of", len(shown))
shown[["section", "difficulty", "domain", "question", "explanation"]].head(10)

## Your turn

Try a different section, difficulty, or number of questions. Then load another small model in Step 2 and compare the scores. Which mistakes seem random, and which reveal a real weakness in the model's reasoning or reading? You can also filter by domain in Step 10 and see whether some topics are harder than others.